In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from pathlib import Path

plt.rcParams.update({
    'font.family': 'serif',
    'font.size': 9,
    'axes.labelsize': 9,
    'axes.titlesize': 9,
    'legend.fontsize': 7.5,
    'xtick.labelsize': 8,
    'ytick.labelsize': 8,
    'lines.linewidth': 1.3,
    'lines.markersize': 4,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'grid.linewidth': 0.5,
    'figure.dpi': 150,
})

BASE   = Path('.')
NS     = [50, 100, 200, 500, 1000]
CYCLE  = 'V'
VMAP_K = 32

SOLVERS = ['ruge_stuben', 'smoothed_aggregation', 'rootnode', 'pairwise']
LABELS  = {
    'ruge_stuben':          'Ruge-Stüben',
    'smoothed_aggregation': 'Smoothed Agg.',
    'rootnode':             'Rootnode',
    'pairwise':             'Pairwise',
}
COLORS  = {
    'ruge_stuben':          '#1f6eb5',
    'smoothed_aggregation': '#d62728',
    'rootnode':             '#2ca02c',
    'pairwise':             '#ff7f0e',
}

In [ ]:
def load_results(*dirs):
    rows = []
    for d in dirs:
        for f in (BASE / d).glob('*.json'):
            data = json.loads(f.read_text())
            rows.append({**data['config'],
                         'time': data['time'],
                         'residual': data['residual']})
    return rows


def get_times(rows, solver, method, device, dtype, mode,
              vmap_k=VMAP_K, cycle=CYCLE, tol=1e-7):
    """Retourne (dof, temps_min) pour chaque n où le solver a convergé (résidu < tol)."""
    subset = [
        r for r in rows
        if r['solver']      == solver
        and r['method']     == method
        and r['device']     == device
        and r['dtype']      == dtype
        and r['mode']       == mode
        and r['cycle_type'] == cycle
        and (mode == 'single' or r['vmap_k'] == vmap_k)
        and r['residual']   < tol          # ← filtre convergence
    ]
    by_n = {}
    for r in subset:
        by_n.setdefault(r['grid_size'], []).append(r['time'])
    ns_ok = [n for n in NS if n in by_n]
    return np.array(ns_ok) ** 2, np.array([min(by_n[n]) for n in ns_ok])


def get_speedup(rows, solver, cpu_method, gpu_method, dtype, mode,
                vmap_k=VMAP_K, cycle=CYCLE, tol=1e-7):
    """Retourne (n, speedup = t_PyAMG / t_AMJax) sur les n où les deux ont convergé."""
    dof_cpu, t_cpu = get_times(rows, solver, cpu_method, 'cpu', dtype, mode, vmap_k, cycle, tol)
    dof_gpu, t_gpu = get_times(rows, solver, gpu_method, 'gpu', dtype, mode, vmap_k, cycle, tol)
    common = np.intersect1d(dof_cpu, dof_gpu)
    if len(common) == 0:
        return np.array([]), np.array([])
    t_c = np.array([t_cpu[dof_cpu == d][0] for d in common])
    t_g = np.array([t_gpu[dof_gpu == d][0] for d in common])
    return np.sqrt(common).astype(int), t_c / t_g   # retourne n (pas n²)


rows = load_results('results_cpu', 'results_gpu_V')

# Vérification rapide des speedups obtenus après filtrage
print('Speedups single | f64 | V-cycle | résidu < 1e-7')
print(f'{"solver":<25} {"pair":<25} {"n":>6}  {"speedup":>8}')
for solver in SOLVERS:
    for cpu_m, gpu_m in [('pyamg','amjax'), ('pyamg_pcg','amjax_pcg')]:
        ns, sp = get_speedup(rows, solver, cpu_m, gpu_m, 'f64', 'single')
        for n, s in zip(ns, sp):
            print(f'{solver:<25} {cpu_m+"/"+gpu_m:<25} {n:>6}  {s:>8.1f}x')

In [ ]:
def make_speedup_figure(mode, vmap_k=VMAP_K, dtype='f64'):
    """mode = 'single' ou 'vmap'."""
    fig, axes = plt.subplots(1, 2, figsize=(7.2, 2.8), sharey=True)

    panels = [
        ('pyamg',     'amjax',     '(a)  AMG standalone'),
        ('pyamg_pcg', 'amjax_pcg', '(b)  AMG-preconditioned CG'),
    ]

    for ax, (cpu_m, gpu_m, title) in zip(axes, panels):
        for solver in SOLVERS:
            dof, speedup = get_speedup(rows, solver, cpu_m, gpu_m, dtype, mode, vmap_k)
            if len(speedup) == 0:
                continue
            # dof = n², on repasse à n pour l'axe x
            ns = np.sqrt(dof).astype(int)
            ax.plot(ns, speedup,
                    color=COLORS[solver], marker='o',
                    label=LABELS[solver])

        ax.axhline(1, color='black', linewidth=0.8, linestyle=':', label='parity')
        ax.set_title(title, pad=4)
        ax.set_xlabel(r'Grid size $n$')
        ax.set_xticks(NS)
        ax.set_xticklabels(NS)
        ax.legend(framealpha=0.8, loc='upper left')

    axes[0].set_ylabel(r'Speedup  $t_{\mathrm{PyAMG}}\,/\,t_{\mathrm{AMJax}}$')

    mode_label = 'single solve' if mode == 'single' else f'batch vmap  (K={vmap_k})'
    fig.suptitle(
        f'Speedup PyAMG / AMJax — {mode_label} — V-cycle — {dtype}',
        fontsize=9, y=1.01
    )
    fig.tight_layout(w_pad=1.5)
    fname = f'fig1_speedup_{mode}.pdf'
    fig.savefig(fname, bbox_inches='tight')
    plt.show()
    print(f'Saved: {fname}')

In [ ]:
make_speedup_figure('single')

In [ ]:
make_speedup_figure('vmap')